# Figure — The highly plastic cell state has the highest intrinsic dimension

**Dataset** [GSE154989](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE154989) —
"Emergence of a high-plasticity cell state during lung cancer evolution", KP/KPT mouse lung
adenocarcinoma time course. 3,891 cells, 12 annotated cell types, 39 mice.

Input: `Input_hpcs_percell.csv`, one row per barcode. All scores 0–1.
Output: `figures/` (600 dpi PNG + vector PDF) and `source_data/`. Provenance in `README.md`.

The HPCS *programme* scored across the SCC EMT states lives in `../Figure_scc_12tumour/`,
with the rest of that cohort's panels.


In [ ]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, rankdata, norm, pearsonr
from statsmodels.stats.multitest import multipletests

FIG, SRC = "figures", "source_data"
os.makedirs(FIG, exist_ok=True); os.makedirs(SRC, exist_ok=True)

DPI = 600
mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": DPI, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.linewidth": 0.9, "axes.labelsize": 12, "axes.titlesize": 13,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
})
sns.set_theme(style="white", context="paper", rc=mpl.rcParams)

# Titles on or off. Journals want bare panels with the caption carrying the text.
SHOW_TITLES = True


def put_title(ax, text, fontsize=13):
    if SHOW_TITLES:
        ax.set_title(text, fontsize=fontsize)


def minmax(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo)


def save(fig, name):
    fig.savefig(f"{FIG}/{name}.png", dpi=DPI)
    fig.savefig(f"{FIG}/{name}.pdf")
    print(f"saved  {FIG}/{name}.png  +  .pdf")


def dunn(data, order):
    """Dunn's post-hoc: pooled ranks, tie correction, BH-FDR, rank-biserial."""
    allv = np.concatenate([data[g] for g in order])
    grp  = np.concatenate([np.repeat(g, len(data[g])) for g in order])
    ranks, N = rankdata(allv), len(allv)
    mean_rank = pd.DataFrame({"g": grp, "r": ranks}).groupby("g")["r"].mean()
    n = {g: len(data[g]) for g in order}
    _, tc = np.unique(allv, return_counts=True)
    tie = 1 - np.sum(tc ** 3 - tc) / (N ** 3 - N)

    def rb(x, y):
        r = rankdata(np.concatenate([x, y]))
        U = r[:len(x)].sum() - len(x) * (len(x) + 1) / 2
        return 2 * U / (len(x) * len(y)) - 1

    rows = []
    for a, b in combinations(order, 2):
        se = np.sqrt((N * (N + 1) / 12) * (1 / n[a] + 1 / n[b]) * tie)
        z = (mean_rank[a] - mean_rank[b]) / se
        rows.append({"group1": a, "group2": b, "z": z, "p": 2 * norm.sf(abs(z)),
                     "median_difference": np.median(data[a]) - np.median(data[b]),
                     "rank_biserial": rb(data[a], data[b])})
    out = pd.DataFrame(rows)
    out["q"] = multipletests(out["p"], method="fdr_bh")[1]
    out["signif"] = pd.cut(out["q"], [-np.inf, 1e-4, 1e-3, 1e-2, 5e-2, np.inf],
                           labels=["****", "***", "**", "*", "ns"])
    return out


---

## 1 · How the ID was calculated

**Per-cell** IDEAS, `method="local_2nn"`, **k = 100**. For each cell the two-nearest-neighbour
estimator is applied to that cell and its 100 nearest neighbours.

k is 100 here and 600 in the SCC and reprogramming figures, because this dataset has only
3,891 cells. Local ID rises with k, so **ID is compared within a dataset, never across
datasets**.

**IDEAS** — <https://github.com/maddalenastn/IDEAS>, vendored in `../src/computeID/`.
Staiano M, Cirone N, Biondo M, Osella M, Scialdone A. *bioRxiv* 2025.
[doi:10.1101/2025.07.21.665922](https://doi.org/10.1101/2025.07.21.665922)

```python
id_result = compute_ID(
    adata,                      # raw counts
    method="local_2nn",
    roots=None,
    n_neighbors=100,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False,
)
```

Run by `../src/run_scripts/run_hpcs_ideas.py` in the `ideas` environment. Raw output:
`../scores/hpcs/IDEAS_local_2NN_GSE154989_k100.csv`.


In [ ]:
PERCELL = "Input_hpcs_percell.csv"
GROUP   = "cell_type"
HPCS    = "HPCS (Cldn4/Slc4a11)"

cells = pd.read_csv(PERCELL, index_col=0)

print(f"{len(cells):,} cells | {cells[GROUP].nunique()} cell types | "
      f"{cells['mouseID'].nunique()} mice")
print(f"raw ID: {cells['ID_score_raw'].min():.2f} - {cells['ID_score_raw'].max():.2f}")
print()
print(cells[GROUP].value_counts().to_string())


---

## 2 · Panel 1 — ID score across cell types

States ordered by median ID; HPCS highlighted. The white box inside each violin is the
interquartile range.


In [ ]:
ORDER = cells.groupby(GROUP, observed=True)["ID_score"].median().sort_values().index.tolist()
LABEL = [s.split(" (")[0] for s in ORDER]
COLOR = ["#C1663A" if s == HPCS else "#4F7FA5" for s in ORDER]

fig, ax = plt.subplots(figsize=(11, 5.5))

sns.violinplot(data=cells, x=GROUP, y="ID_score", order=ORDER,
               inner=None, cut=0, linewidth=1.0, palette=COLOR, hue=GROUP,
               hue_order=ORDER, legend=False, ax=ax)

sns.boxplot(data=cells, x=GROUP, y="ID_score", order=ORDER,
            width=0.14, showfliers=False,
            boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.9),
            whiskerprops=dict(color="black", linewidth=0.9),
            capprops=dict(color="black", linewidth=0.9),
            medianprops=dict(color="black", linewidth=1.2), ax=ax)

ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels(LABEL, rotation=45, ha="right", fontsize=9)
ax.set_xlabel("Cell type", fontsize=12)
ax.set_ylabel("ID score", fontsize=12)
put_title(ax, "ID score across cell types")
sns.despine(ax=ax)
fig.tight_layout()
save(fig, "panel01_id_across_cell_types")
plt.show()

summary = (cells.groupby(GROUP, observed=True)["ID_score"]
           .agg(["count", "mean", "median", "std"]).reindex(ORDER).round(4))
print(summary.to_string())
summary.to_csv(f"{SRC}/panel01_id_across_cell_types.csv")


---

## 3 · Statistics — HPCS against every other cell type

Kruskal-Wallis across all twelve, then Dunn with BH-FDR. The eleven comparisons that involve
HPCS are pulled out, with the median ID difference as the effect size.


In [ ]:
data = {g: cells.loc[cells[GROUP] == g, "ID_score"].values for g in ORDER}
H, p_kw = kruskal(*[data[g] for g in ORDER])
print(f"Kruskal-Wallis:  H = {H:.2f}   p = {p_kw:.3e}\n")

tab = dunn(data, ORDER)
tab.to_csv(f"{SRC}/panel01_statistics_all_pairs.csv", index=False)

hp = (tab[(tab["group1"] == HPCS) | (tab["group2"] == HPCS)]
      .assign(other=lambda t: np.where(t["group1"] == HPCS, t["group2"], t["group1"]),
              diff=lambda t: np.where(t["group1"] == HPCS,
                                      t["median_difference"], -t["median_difference"]))
      .loc[:, ["other", "z", "p", "q", "diff", "rank_biserial", "signif"]]
      .sort_values("q"))
hp["other"] = [o.split(" (")[0] for o in hp["other"]]

print("HPCS vs the other 11 cell types:\n")
print(hp.to_string(index=False))
hp.to_csv(f"{SRC}/panel02_hpcs_vs_others.csv", index=False)

n_sig = int((hp["q"] < 0.05).sum())
print(f"\n{n_sig}/11 significant at BH-FDR < 0.05, largest q = {hp['q'].max():.4f}")


---

## 4 · Panel 2 — HPCS effect size

How much higher the median ID is in HPCS than in each other cell type, largest first.


In [ ]:
h = hp.sort_values("diff")
n_sig = int((hp["q"] < 0.05).sum())

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(h["other"], h["diff"], color="#C1663A", height=0.65)
for bar, q, d in zip(bars, h["q"], h["diff"]):
    ax.text(d + 0.004, bar.get_y() + bar.get_height() / 2,
            f"{d:.3f}" + ("" if q < 0.05 else "   ns"),
            va="center", fontsize=9)

ax.set_xlabel("Median ID-score difference (HPCS − cell type)", fontsize=12)
ax.set_ylabel("Cell type", fontsize=12)
put_title(ax,
    "HPCS is higher than every other cell type\n"
    f"{n_sig}/11 pairwise comparisons significant after FDR correction "
    f"(largest q = {hp['q'].max():.4f})", 12)
ax.set_xlim(0, h["diff"].max() * 1.22)
sns.despine(ax=ax)
fig.tight_layout()
save(fig, "panel02_hpcs_effect_size")
plt.show()


---

## 5 · Source data

| File | Contents |
|---|---|
| `panel01_id_across_cell_types.csv` | ID: count, mean, median, sd per cell type |
| `panel01_statistics_all_pairs.csv` | Dunn, all 66 pairs |
| `panel02_hpcs_vs_others.csv` | the 11 HPCS comparisons with effect sizes |
